In [71]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Madhurjya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Madhurjya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [72]:
import pandas as pd

true_df=pd.read_csv("True.csv")
fake_df=pd.read_csv("Fake.csv")
true_df['label']='REAL'
fake_df['label']='FAKE'

In [73]:
real_sample=true_df.sample(n=2000, random_state=42)
fake_sample=fake_df.sample(n=2000, random_state=42)

reduced_df=pd.concat([real_sample, fake_sample]).sample(frac=1, random_state=42).reset_index(drop=True)
print(reduced_df['label'].value_counts())
print(reduced_df.head())

label
REAL    2000
FAKE    2000
Name: count, dtype: int64
                                               title  \
0  May's government pushes Brexit bill to avoid '...   
1   Trump’s EPA OKs Pesticide That Causes Brain D...   
2  Man arrested at Trump rally said he wanted to ...   
3  Jared Kushner NEVER Registered To Vote As A “F...   
4  MARTHA STEWART Makes Lewd Gesture Towards Trum...   

                                                text       subject  \
0  LONDON (Reuters) - Brexit minister David Davis...     worldnews   
1  Farmworkers were pulled from fields on Friday ...          News   
2  (Reuters) - A man arrested over the weekend tr...  politicsNews   
3  Meanwhile, as President Trump continues to mee...     left-news   
4  Martha, Martha, Martha You re 75-years old! Ti...     left-news   

                 date label  
0  September 6, 2017   REAL  
1        May 15, 2017  FAKE  
2      June 20, 2016   REAL  
3        Sep 29, 2017  FAKE  
4         May 8, 2017  FAKE  


In [74]:
reduced_df['content']=reduced_df['title']+ " "+reduced_df['text']
reduced_df=reduced_df[['content', 'label']]
reduced_df.head()

,content,label
0,May's government pushes Brexit bill to avoid '...,REAL
1,Trump’s EPA OKs Pesticide That Causes Brain D...,FAKE
2,Man arrested at Trump rally said he wanted to ...,REAL
3,Jared Kushner NEVER Registered To Vote As A “F...,FAKE
4,MARTHA STEWART Makes Lewd Gesture Towards Trum...,FAKE


In [75]:
from sklearn.preprocessing import LabelEncoder

label=LabelEncoder()
y_encoded=label.fit_transform(reduced_df['label'])
print(label.classes_)

['FAKE' 'REAL']


In [76]:
import re
import nltk
nltk.download ('stopwords')

from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Madhurjya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [77]:
stop_words=set(stopwords.words('english'))
def clean_text(text):
    text=text.lower()
    text=re.sub(r'[^a-z\s]', '', text)
    words=text.split()
    words=[w for w in words if w not in stop_words]
    return " ".join(words)
reduced_df['clean_text']=reduced_df['content'].apply(clean_text)


In [78]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words=1000
max_len=100

token=Tokenizer(num_words=max_words, oov_token="<00V>")
token.fit_on_texts(reduced_df['clean_text'])
sequences=token.texts_to_sequences(reduced_df['clean_text'])
pad=pad_sequences(sequences, maxlen=max_len, padding='post')

In [79]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test=train_test_split(pad, y_encoded, test_size=0.2, random_state=42)

In [80]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Conv1D, Dense, Dropout, Softmax, Multiply, Lambda
import tensorflow.keras.backend as K

In [81]:
# Input layer
inputs = Input(shape=(max_len,))

# Embedding
x = Embedding(input_dim=max_words, output_dim=128)(inputs)

# CNN layer
x = Conv1D(filters=128, kernel_size=5, activation='relu')(x)

# ------------------ ATTENTION STARTS HERE ------------------

# Step 1: Score each time step (word position)
attention = Dense(64, activation='tanh')(x)   # (batch, time_steps, 1)
attention = Dense(1)(attention)   # (batch, time_steps, 1)

# Step 2: Flatten to compute weights
attention = Lambda(lambda x: K.squeeze(x, axis=-1))(attention)  # (batch, time_steps)

# Step 3: Softmax → normalize weights
attention = Softmax(name="attention_weights")(attention)  # weights sum to 1

# Step 4: Expand dimensions
attention = Lambda(lambda x: K.expand_dims(x, axis=-1))(attention)  # (batch, time_steps, 1)

# Step 5: Apply weights
x = Multiply()([x, attention])

# Step 6: Sum over time dimension
x = Lambda(lambda x: K.sum(x, axis=1))(x)

# ------------------ ATTENTION ENDS HERE ------------------

# Dense layers
x = Dropout(0.5)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.5)(x)

# Output
outputs = Dense(1, activation='sigmoid')(x)

# Final model
model = Model(inputs, outputs)

model.summary()

Model: "functional_10"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_6         │ (None, 100, 128)  │    128,000 │ input_layer_6[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_6 (Conv1D)   │ (None, 96, 128)   │     82,048 │ embedding_6[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_20 (Dense)    │ (None, 96, 64)    │      8,256 │ conv1d_6[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_21 (Dense)    │ (None, 96, 1)     │         65 │ dense_20[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_15 (Lambda)  │ (None, 96)        │          0 │ dense_21[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_weights   │ (None, 96)        │          0 │ lambda_15[0][0]   │
│ (Softmax)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_16 (Lambda)  │ (None, 96, 1)     │          0 │ attention_weight… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_5          │ (None, 96, 128)   │          0 │ conv1d_6[0][0],   │
│ (Multiply)          │                   │            │ lambda_16[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_17 (Lambda)  │ (None, 128)       │          0 │ multiply_5[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 128)       │          0 │ lambda_17[0][0]   │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_22 (Dense)    │ (None, 64)        │      8,256 │ dropout_12[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_13          │ (None, 64)        │          0 │ dense_22[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_23 (Dense)    │ (None, 1)         │         65 │ dropout_13[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 226,690 (885.51 KB)

 Trainable params: 226,690 (885.51 KB)

 Non-trainable params: 0 (0.00 B)

In [82]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

history = model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.6648 - loss: 0.5909 - val_accuracy: 0.8391 - val_loss: 0.3617
Epoch 2/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9035 - loss: 0.2574 - val_accuracy: 0.9250 - val_loss: 0.1934
Epoch 3/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9496 - loss: 0.1449 - val_accuracy: 0.9312 - val_loss: 0.1890
Epoch 4/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9703 - loss: 0.0917 - val_accuracy: 0.9219 - val_loss: 0.2050
Epoch 5/15
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9840 - loss: 0.0504 - val_accuracy: 0.9219 - val_loss: 0.2709


In [83]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", test_accuracy)

25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9300 - loss: 0.2088
Test Accuracy: 0.9300000071525574


In [84]:
attention_model = Model(inputs=model.input,
  outputs=model.get_layer("attention_weights").output)

In [85]:
sample_text = reduced_df['clean_text'][0]

seq = token.texts_to_sequences([sample_text])
pad_seq = pad_sequences(seq, maxlen=max_len, padding='post')

attention = attention_model.predict(pad_seq)[0]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step


In [86]:
words = sample_text.split()

# Trim attention to actual sentence length
attention = attention[:len(words)]

for word, score in zip(words, attention):
    print(f"{word}: {score:.4f}")

mays: 0.0003
government: 0.0003
pushes: 0.0025
brexit: 0.0024
bill: 0.0014
avoid: 0.0010
chaotic: 0.0006
departure: 0.0003
london: 0.0007
reuters: 0.0018
brexit: 0.1843
minister: 0.0094
david: 0.3073
davis: 0.1012
called: 0.0172
parliament: 0.0076
thursday: 0.0013
back: 0.0009
legislation: 0.0007
sever: 0.0004
britain: 0.0004
political: 0.0005
financial: 0.0030
legal: 0.0016
ties: 0.0003
european: 0.0004
union: 0.0003
saying: 0.0003
opposing: 0.0007
bill: 0.0014
would: 0.0014
lead: 0.0134
chaos: 0.0050
rowdy: 0.0007
session: 0.0006
parliament: 0.0007
davis: 0.0012
accused: 0.0006
opposition: 0.0010
labour: 0.0177
party: 0.0013
pursuing: 0.0007
cynical: 0.0007
unprincipled: 0.0006
path: 0.0018
challenging: 0.0018
repeal: 0.0455
bill: 0.0073
eu: 0.0115
withdrawal: 0.0015
bill: 0.0015
designed: 0.0008
disentangle: 0.0011
britain: 0.0878
years: 0.0013
eu: 0.0028
lawmaking: 0.0460
labour: 0.0010
turn: 0.0004
said: 0.0010
government: 0.0012
using: 0.0007
bill: 0.0017
give: 0.0018
wideranging

In [87]:
top_k = 8
top_indices = attention.argsort()[-top_k:]

for i in top_indices:
    print(f"{words[i]} → {attention[i]:.4f}")

march → 0.0197
eu → 0.0428
repeal → 0.0455
lawmaking → 0.0460
britain → 0.0878
davis → 0.1012
brexit → 0.1843
david → 0.3073


In [88]:
for word, score in zip(words, attention):
    if score > 0.05:
        print(f"[{word.upper()}]", end=" ")
    else:
        print(word, end=" ")

mays government pushes brexit bill avoid chaotic departure london reuters [BREXIT] minister [DAVID] [DAVIS] called parliament thursday back legislation sever britain political financial legal ties european union saying opposing bill would lead chaos rowdy session parliament davis accused opposition labour party pursuing cynical unprincipled path challenging repeal bill eu withdrawal bill designed disentangle [BRITAIN] years eu lawmaking labour turn said government using bill give wideranging powers blank cheque away laws ministers like threatening rights ordinary britons legislation vital steppingstone towards britain departure eu march faces stormy debate likely barrage attempted amendments prime minister theresa may weakened 

In [89]:
def show_attention(index):
    sample_text = reduced_df['clean_text'][index]
    label = reduced_df.iloc[index]['label']

    seq = token.texts_to_sequences([sample_text])
    pad_seq = pad_sequences(seq, maxlen=max_len, padding='post')

    attention = attention_model.predict(pad_seq)[0]

    words = sample_text.split()
    attention = attention[:len(words)]

    print("\n==============================")
    print("Actual Label:", label)
    print("Text:\n", sample_text[:200], "...\n")

    # Top-K words
    top_k = 8
    top_indices = attention.argsort()[-top_k:]

    print("Top Important Words:")
    for i in top_indices:
        print(f"{words[i]} → {attention[i]:.4f}")

In [90]:
fake_indices = reduced_df[reduced_df['label'] == 'FAKE'].index.tolist()
real_indices = reduced_df[reduced_df['label'] == 'REAL'].index.tolist()

In [98]:
show_attention(fake_indices[0])
show_attention(fake_indices[10])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step

Actual Label: FAKE
Text:
 trumps epa oks pesticide causes brain damage children farmworkers getting sick farmworkers pulled fields friday came contact chemical epa recently approved chemical originally flagged obama administra ...

Top Important Words:
caught → 0.0028
dan → 0.0039
chemical → 0.0042
symptoms → 0.0559
observedtwelve → 0.0937
nausea → 0.1609
suffered → 0.2571
people → 0.3704
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step

Actual Label: FAKE
Text:
 birmingham tells alabama gop go fk passes highest minimum wage south alabama republicans quickly trying ban cities counties raising minimum wage one city gave gop finger beat punchbirmingham alabama h ...

Top Important Words:
quickly → 0.0112
faulkner → 0.0115
alabama → 0.0120
introduced → 0.0133
entirely → 0.0293
bills → 0.0783
food → 0.2731
put → 0.4370


In [99]:
show_attention(real_indices[0])
show_attention(real_indices[10])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step

Actual Label: REAL
Text:
 mays government pushes brexit bill avoid chaotic departure london reuters brexit minister david davis called parliament thursday back legislation sever britain political financial legal ties european  ...

Top Important Words:
march → 0.0197
eu → 0.0428
repeal → 0.0455
lawmaking → 0.0460
britain → 0.0878
davis → 0.1012
brexit → 0.1843
david → 0.3073
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step

Actual Label: REAL
Text:
 kurd forces move back defense line around kirkuk disengagement effort kirkuk iraq reuters kurdish forces moved back defensive line around oil region kirkuk northern iraq km miles reduce possibility fr ...

Top Important Words:
security → 0.0118
iraqi → 0.0132
said → 0.0374
region → 0.0415
effort → 0.0529
kirkuk → 0.0849
iraq → 0.1201
reuters → 0.5814


In [92]:
import random

show_attention(random.choice(fake_indices))
show_attention(random.choice(real_indices))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step

Actual Label: FAKE
Text:
 trump administration helping scam universities steal money students trump department education delaying action requests student loan forgiveness whose money stolen scams forprofit universities work er ...

Top Important Words:
work → 0.0046
action → 0.0048
obama → 0.0054
million → 0.0545
million → 0.0929
stolen → 0.1500
students → 0.2541
loans → 0.3456
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step

Actual Label: REAL
Text:
 house senate republicans face challenge corporate amt tax washington reuters us republicans congress grappling thorny question corporate taxes work reconcile competing tax bills senate house represent ...

Top Important Words:
amt → 0.0219
calling → 0.0270
already → 0.0337
corporate → 0.0532
minimum → 0.0533
repealing → 0.0667
top → 0.2614
estate → 0.3094


In [93]:
print(reduced_df['label'].value_counts())

label
REAL    2000
FAKE    2000
Name: count, dtype: int64


In [94]:
print(real_indices[:10])

[0, 2, 5, 6, 7, 8, 9, 12, 13, 16]


In [95]:
for i in real_indices[:5]:
    print(i, reduced_df['label'][i])

0 REAL
2 REAL
5 REAL
6 REAL
7 REAL


In [96]:
# Save CNN model
model.save("cnn.keras")

print("CNN model saved successfully!")

CNN model saved successfully!


In [97]:
import joblib

# Save tokenizer
joblib.dump(token, "cnn_tokenizer.pkl")

print("Tokenizer saved successfully!")

Tokenizer saved successfully!
